# Logistic Regression Exercise

Now it's your turn to implement logistic regression on a new data set. For this purpose we use the Titanic Dataset. It includes personal information of all passengers on the Titanic as well as they survived the sinking of the Titanic or died.

Here’s the **Data Dictionary** of the dataset:

- PassengerID: type should be integers

- Survived: survived or not

- Pclass: class of Travel of every passenger

- Name: the name of the passenger

- Sex: gender

- Age: age of passengers

- SibSp: No. of siblings/spouse aboard

- Parch: No. of parent/child aboard

- Ticket: Ticket number

- Fare: what Prices they paid

- Cabin: cabin number

- Embarked: the port in which a passenger has embarked.

        - C: Cherbourg , S: Southampton , Q: Queenstown


You will find the data in the data folder.


## What you should do:

- conduct a brief EDA to become familiar with the data
- use Logistic Regression to predict if a passenger died or not

## How to do it:

Time is short, so aim for the simplest viable product first:
1. Load the data

2. Separate features and target 

3. Split the data in train and test

3. Get a quick overview of the train data

4. Agree on a classification metric for the task 

5. Create a simple heuristic/educated guess for the classification first. This is called a "baseline model". It is used to compare more complex models later (in this case: logistic regression). You as a data scientist want to prove how much your work/ML could improve the business metric, therefore you need a baseline model for comparison. In some cases you want to improve on an already existing model in your company which would be your baseline model then. In other cases, there are typical baseline models used in the specific field. For other tasks, you have to come up with a simple but meaningful idea, how to classify the data based on your business understanding (EDA). A baseline model should follow Occam’s Razor principle: "A simple model is the best model". 
    - Example of a baseline model: 
    If the task is to classify cats and dogs, a baseline model could be: We classify every animal as cat if its weight < 5 kg, otherwise the animal is classified as a dog. (The value of 5 kg is an educated guess, based on our business understanding/EDA.) 

6. use one or two already numerical features to create a simple first model
    -  did it even beat your base model?

7. Now you can go through the data science lifecycle again and again:
    - clean the data better

    - get more insights with EDA

    - add more features

    - do feature engineering 
    
    and check if your work improves your model further!

8. Stop whenever time is up or you cannot improve your model any further.

This repository contains a solution to this problem. If you want to compare your final result with the result of this repository solution, choose **25** as random seed and a test size of 30% for your train test split.

In [1]:
## Load the data
import pandas as pd

df = pd.read_csv("data/titanic.csv")
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
## Separate features and target
# Target: what we want to predict
y = df["Survived"]

# Features: everything else, minus columns that are just identifiers
# (PassengerId, Name, Ticket) or mostly missing (Cabin)
X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])

print(X.shape, y.shape)
X.head()

(891, 7) (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [3]:
## Split the data between train & test
from sklearn.model_selection import train_test_split

# 30% test size and random seed 25, as suggested, so results are comparable with the solution.
# stratify=y keeps the same survived/died ratio in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Survival rate train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

Train: (623, 7) (623,)
Test:  (268, 7) (268,)
Survival rate train: 0.384 | test: 0.384


In [4]:
## Overview of the train data
# Put features and target back together, only for exploring (the test set stays untouched)
train = X_train.assign(Survived=y_train)

# Column types and non-null counts
train.info()

# Missing values per column
print("\nMissing values:")
print(train.isna().sum())

<class 'pandas.DataFrame'>
Index: 623 entries, 468 to 745
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    623 non-null    int64  
 1   Sex       623 non-null    str    
 2   Age       499 non-null    float64
 3   SibSp     623 non-null    int64  
 4   Parch     623 non-null    int64  
 5   Fare      623 non-null    float64
 6   Embarked  622 non-null    str    
 7   Survived  623 non-null    int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 43.8 KB

Missing values:
Pclass        0
Sex           0
Age         124
SibSp         0
Parch         0
Fare          0
Embarked      1
Survived      0
dtype: int64


In [5]:
# Summary statistics: numeric columns
display(train.describe())

# Categorical columns: how often each value appears
for col in ["Sex", "Embarked", "Pclass"]:
    print(train[col].value_counts(dropna=False), "\n")

,Pclass,Age,SibSp,Parch,Fare,Survived
count,623.000000,499.000000,623.000000,623.000000,623.000000,623.000000
mean,2.324238,29.537255,0.524880,0.361156,32.090221,0.383628
std,0.831943,14.278981,1.126532,0.751996,50.732210,0.486660
min,1.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,20.000000,0.000000,0.000000,7.895800,0.000000
50%,3.000000,28.000000,0.000000,0.000000,13.500000,0.000000
75%,3.000000,37.500000,1.000000,0.000000,30.000000,1.000000
max,3.000000,71.000000,8.000000,5.000000,512.329200,1.000000


Sex
male      409
female    214
Name: count, dtype: int64 

Embarked
S      457
C      112
Q       53
NaN      1
Name: count, dtype: int64 

Pclass
3    349
1    147
2    127
Name: count, dtype: int64 



In [6]:
# Survival rate per group: which features seem to matter?
for col in ["Sex", "Pclass", "Embarked", "SibSp", "Parch"]:
    print(train.groupby(col)["Survived"].agg(["mean", "count"]).round(3), "\n")

# Numeric features: average value for survivors vs. non-survivors
display(train.groupby("Survived")[["Age", "Fare", "SibSp", "Parch"]].mean().round(2))

         mean  count
Sex                 
female  0.752    214
male    0.191    409 

         mean  count
Pclass              
1       0.646    147
2       0.449    127
3       0.249    349 

           mean  count
Embarked              
C         0.554    112
Q         0.396     53
S         0.339    457 

        mean  count
SibSp              
0      0.352    432
1      0.533    135
2      0.409     22
3      0.231     13
4      0.273     11
5      0.000      5
8      0.000      5 

        mean  count
Parch              
0      0.343    478
1      0.532     79
2      0.525     59
3      0.667      3
4      0.000      1
5      0.000      3 



,Age,Fare,SibSp,Parch
Survived,,,,
0,30.31,21.14,0.56,0.30
1,28.39,49.68,0.46,0.46


**Metric choice: F1-score (positive class = survived), with accuracy + confusion matrix as context.**

- The classes are imbalanced: ~38% survived, ~62% died. A "model" that always predicts *died* already gets ~62% accuracy, so accuracy alone flatters weak models.
- F1 combines precision (of those we predict survived, how many really did) and recall (of those who survived, how many we find). The "always died" model scores F1 = 0, so F1 shows whether a model actually learned something.
- Missing a survivor and raising a false alarm cost about the same here, so we don't favour precision or recall and use their balance, F1.
- We still report accuracy (easy to explain, compared against the ~62% you get for free) and the confusion matrix (to see where errors happen).

In [7]:
## Agree on a classification metric for the task
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

def evaluate(y_true, y_pred, name="model"):
    """Print the metrics we agreed on: F1 (main), accuracy, confusion matrix."""
    print(f"--- {name} ---")
    print("F1 (survived=1):", round(f1_score(y_true, y_pred, zero_division=0), 3))
    print("Accuracy:       ", round(accuracy_score(y_true, y_pred), 3))
    print("Confusion matrix [rows=true 0/1, cols=pred 0/1]:")
    print(confusion_matrix(y_true, y_pred))

# Why not accuracy alone? Predicting "died" for everyone already looks decent:
evaluate(y_train, [0] * len(y_train), name="always predict 'died'")

--- always predict 'died' ---
F1 (survived=1): 0.0
Accuracy:        0.616
Confusion matrix [rows=true 0/1, cols=pred 0/1]:
[[384   0]
 [239   0]]


In [8]:
## Baseline model (-> create a simple / educated guess)
# Educated guess from the EDA: "women and children first".
# In the train data 75% of women survived but only 19% of men,
# so the simplest rule is: predict survived if female, died if male.
def baseline_predict(X):
    return (X["Sex"] == "female").astype(int)

# Train: the data the rule was derived from
evaluate(y_train, baseline_predict(X_train), name="baseline (female -> survived) | train")

# Test: the score our logistic regression has to beat
evaluate(y_test, baseline_predict(X_test), name="baseline (female -> survived) | test")

--- baseline (female -> survived) | train ---
F1 (survived=1): 0.711
Accuracy:        0.79
Confusion matrix [rows=true 0/1, cols=pred 0/1]:
[[331  53]
 [ 78 161]]
--- baseline (female -> survived) | test ---
F1 (survived=1): 0.709
Accuracy:        0.78
Confusion matrix [rows=true 0/1, cols=pred 0/1]:
[[137  28]
 [ 31  72]]


Picked **Pclass** and **Fare** for a first model because:
- they are already numerical, so no encoding is needed,
- they have **no missing values** (unlike Age, which is missing for ~20% of passengers),
- the EDA showed both are linked to survival: 65% survived in 1st class vs. 25% in 3rd, and survivors paid on average ~50 vs. ~21 for non-survivors.

In [ ]:
## First model
# Logistic regression on two already numerical features: Pclass and Fare
from sklearn.linear_model import LogisticRegression

features = ["Pclass", "Fare"]

log_reg = LogisticRegression()
log_reg.fit(X_train[features], y_train)

# Coefficients: negative for Pclass (higher class number -> lower survival chance),
# positive for Fare (higher fare -> higher survival chance)
print("Coefficients:", dict(zip(features, log_reg.coef_[0].round(3))))
print("Intercept:   ", log_reg.intercept_.round(3), "\n")

evaluate(y_train, log_reg.predict(X_train[features]), name="log. regression (Pclass, Fare) | train")
evaluate(y_test, log_reg.predict(X_test[features]), name="log. regression (Pclass, Fare) | test")

**Did it beat the baseline? No.**

| Model (test set) | F1 | Accuracy |
|---|---|---|
| Baseline (female -> survived) | 0.709 | 0.780 |
| Logistic regression (Pclass, Fare) | 0.477 | 0.664 |

The model is very cautious: it predicts "survived" only for a few passengers (mostly 1st class / high fare) and misses most survivors (62 of 103). Class and fare alone carry much less information than **Sex**, the strongest signal we found in the EDA. Next step: add Sex (encoded as 0/1) and Age (after filling missing values) as features.

## Second model: add Sex and Age

Two preparation steps before we can use them:
- **Sex** is text, encoded it as a number: `female = 1`, `male = 0`.
- **Age** is missing for ~20% of passengers. Filled the gaps with the **median age of the train set**. Computeed the median on train only and reuse it for test, so no information from the test set leaks into the model. The median is used instead of the mean because it is less affected by a few very old passengers.

In [9]:
## Prepare Sex and Age
# Median age computed on the TRAIN data only (reused for test -> no data leakage)
age_median = X_train["Age"].median()
print("Median age (train):", age_median)

def prepare(X):
    """Encode Sex as 0/1 and fill missing Age with the train median."""
    X = X.copy()                                     # don't change the original data
    X["Sex"] = (X["Sex"] == "female").astype(int)    # female = 1, male = 0
    X["Age"] = X["Age"].fillna(age_median)
    return X

X_train_prep = prepare(X_train)
X_test_prep = prepare(X_test)

# Check: no missing values left in the features we use
print(X_train_prep[["Pclass", "Fare", "Sex", "Age"]].isna().sum())

Median age (train): 28.0
Pclass    0
Fare      0
Sex       0
Age       0
dtype: int64


In [11]:
## Second model
# Logistic regression on Pclass, Fare, Sex and Age
features_2 = ["Pclass", "Fare", "Sex", "Age"]

log_reg_2 = LogisticRegression(max_iter=1000)
log_reg_2.fit(X_train_prep[features_2], y_train)

# Coefficients: Sex is by far the strongest (being female raises the survival chance a lot),
# Age is negative (older -> slightly lower survival chance)
print("Coefficients:", dict(zip(features_2, log_reg_2.coef_[0].round(3))))
print("Intercept:   ", log_reg_2.intercept_.round(3), "\n")

evaluate(y_train, log_reg_2.predict(X_train_prep[features_2]), name="log. regression (Pclass, Fare, Sex, Age) | train")
evaluate(y_test, log_reg_2.predict(X_test_prep[features_2]), name="log. regression (Pclass, Fare, Sex, Age) | test")

NameError: name 'LogisticRegression' is not defined

**Did it beat the baseline? Yes, on our main metric (F1).**

| Model (test set) | F1 | Accuracy |
|---|---|---|
| Baseline (female -> survived) | 0.709 | 0.780 |
| Logistic regression (Pclass, Fare) | 0.477 | 0.664 |
| Logistic regression (Pclass, Fare, Sex, Age) | **0.725** | 0.776 |

- F1 improved from 0.709 to 0.725: the model finds more survivors than the baseline (79 of 103 instead of 72), at the cost of a few more false alarms (36 instead of 28).
- Accuracy is about the same as the baseline (0.776 vs. 0.780), so the gain is small. Sex still does most of the work; Pclass and Age add some extra information (e.g. women in 3rd class survive less often).
- Train and test scores are close (F1 0.729 vs. 0.725), so the model is not overfitting.

Possible next steps: add Embarked (one-hot encoded), SibSp/Parch (e.g. a "family size" feature), scale the features, or tune the decision threshold.